In [ ]:
import CDutils
import scipy.io as sio
import numpy as np
from scipy.io import savemat
import os
import nolds
import data_util as du
import exact_util as eu

import matplotlib.pyplot as plt
from config import Config, get_cfg_defaults
from sklearn.metrics import f1_score

In [ ]:
path = r"D:\Contraction_detection_publication\data"
dir_list = os.listdir(path)

In [ ]:
dir_list

In [ ]:
def sample_entropy(signal):
    # global global_threshold
    # T = global_threshold
    T = np.std(signal)
    sampen = nolds.sampen(signal, tolerance=0.25*T)
    return sampen


def root_mean_square(signal):
    rms = np.sqrt(1/len(signal) * np.sum(np.power(signal,2)))
    return rms


def curve(signal,method, fs=4, window_size=60, shift= 1,mean_filter_size = 10):
    
    y = eu.get_curve(signal, window_size=window_size, shift = shift, method=method)#zero_crossing_1
#     if mean_filter_size != 0:
#         DataFilter.perform_rolling_filter(y, mean_filter_size*fs, AggOperations.MEAN.value)

    return y

def analysis(signal,method):
    
    window_size, shift,base_window_size,base_shift, mean_filter_size = 60,20, 180, 20, 10

    global_threshold =  np.std(signal)
    fs = 4
    y_origin = curve(signal,method, fs, window_size=window_size*fs, shift= fs*shift, mean_filter_size = mean_filter_size)

    iup = eu.remove_base(y_origin, base_window_size = base_window_size*fs, shift = fs*10)

    onsets,offsets, ans = detect_edge(iup, 5*fs, weight = bound_weight, fs = fs, window_size = window_size)
    
    return ans


def detect_edge(signal, d, weight, fs, window_size):
    def judge(arr, th, on = True):
        flag = 0
        for i in range(len(arr)):
            if on and arr[i] >= th:
                flag += 1
            elif on is not True and arr[i] < th:
                flag += 1
        return flag == len(arr)
    s = 1
    onsets, offsets,ans = [],[],[]
    
    signal_ = signal[fs*(window_size+1):-fs*(window_size+1)]
    
    f_min = np.min(signal_)
    f_max = np.max(signal_)

    th = f_min+weight*(f_max-f_min)
   
    flag = False
    i = 0
    while i < len(signal_):
        tmp = signal_[i:i+d]
        if judge(tmp,th, True) and flag is not True:
                flag = True
                onsets.append(i)
        if judge(tmp,th, False) and flag:
                flag = False
                offsets.append(i)
                ans.append(onsets[-1] + np.argmax(signal_[onsets[-1]:i]))
                #i+=fs
        i += s
    if len(onsets)> len(offsets):
        if onsets[-1] != len(signal_)-1:
            offsets.append(len(signal_)-1)
        else:
            onsets = onsets[:len(offsets)] 
    onsets = np.array(onsets)+fs*window_size+fs
    offsets = np.array(offsets)+fs*window_size+fs
    ans = np.array(ans)+fs*window_size+fs
    return onsets, offsets, ans

In [ ]:
global_threshold = 0

cfg = get_cfg_defaults()
fs = cfg.PARAM.FS
channel = cfg.PARAM.CHANNEL
filter = cfg.PARAM.FILTER
method =sample_entropy
cfg.DETECT.METHOD_NAME = method.__name__
method_name = cfg.DETECT.METHOD_NAME 
bound_weight= 0.2
print(cfg)

In [ ]:
f1_list = []
cp_score_list = []
cpn_list = []
for pt in range(len(dir_list)):
    data = sio.loadmat(path + '/' + dir_list[pt])
    modulation_ds = data['tmp_y']
    label_ds = data['newClusterSig']
    connectivity = data['Uterus']['tri'][0][0]
    conn = connectivity.astype(int)-1
    
    ch_num = modulation_ds.shape[0]
    conn_num = conn.shape[0]
    signal_length = modulation_ds.shape[1]
    
    modulation_ar_removal = CDutils.ar_remove_patient(modulation_ds)
    
    c_detection = np.zeros_like(modulation_ar_removal)
    for ch in range(ch_num):
        sig = modulation_ar_removal[ch,:]
        c = np.zeros(sig.shape[0])
        result = analysis(sig,method)
        result = np.insert(result, 0, 0., axis=0)
        for i in range(result.shape[0]//2):
            c[result[2*i]:result[2*i+1]] = 1
        c_detection[ch,:] = c 

    f1_array = np.zeros(ch_num,)
    cp_score_array = np.zeros(ch_num,)
    cpn_array = np.zeros(ch_num,)
    
    for ch in range(ch_num):
        pre_cpds = CDutils.find_cpd(c_detection[ch,:], 1)
        gt_cpds = CDutils.find_cpd(label_ds[ch,:], 1)
        f1_array[ch] = f1_score(label_ds[ch,:], c_detection[ch,:], zero_division=1.0)
        cp_score_array[ch] = CDutils.cp_score(gt_cpds,pre_cpds,signal_length)
        # cp_score_array1[ch] = CDutils.cp_score1(gt_cpds,pre_cpds,signal_length)
        cpn_array[ch] = (np.abs(len(gt_cpds)-len(pre_cpds)))/len(gt_cpds)
    print(np.median(f1_array))
    print(np.median(cp_score_array))
    print(np.median(cpn_array))
    print('-----')
    f1_list.append(f1_array)
    cp_score_list.append(cp_score_array)
    cpn_list.append(cpn_array)

In [ ]:
mdic = {"f1_list": f1_list,"cp_score_list": cp_score_list,"cp_score1_list": cp_score1_list}
savemat(r"D:\Code_Hansong\Contraction detection manuscript\Contaction_detection\results\8_sample_entropy_result.mat", mdic)